# M2 · Prompt Ensemble Bahasa Indonesia
**IF25-40304 Pembelajaran Mesin Multimodal · Kelompok 05**


## 0 · Install dependensi (sekali saja)

In [1]:
# !pip install sentence-transformers numpy pandas


## 1 · Konfigurasi dan templat

In [2]:
import json, time, numpy as np

SEED  = 0
N_SUB = 500   # ukuran subset train untuk seleksi greedy
MAX_K = 5     # batas maksimum templat greedy

TEMPLATES_ID = [
    "{}",
    "foto {}",
    "sebuah foto {}",
    "gambar {}",
    "foto budaya Indonesia: {}",
    "dokumentasi budaya Indonesia: {}",
    "{} di Indonesia",
    "sebuah foto yang menunjukkan {}",
    "potret {}",
    "ini adalah foto {}",
]
TEMPLATES_EN = [
    "{}",
    "a photo of {}",
    "a picture of {}",
    "a photo showing {}",
    "a photo of Indonesian culture: {}",
    "documentation of Indonesian culture: {}",
    "{} in Indonesia",
    "this is a photo of {}",
]
print(f"Templat ID: {len(TEMPLATES_ID)} | Templat EN: {len(TEMPLATES_EN)} | seed={SEED}")


Templat ID: 10 | Templat EN: 8 | seed=0


## 2 · Fungsi utilitas

In [3]:
def l2norm(x):
    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12)

def apply_template(tpl, text, lower_first=True):
    if lower_first and tpl != "{}" and text and text[0].isupper() and not text[:2].isupper():
        text = text[0].lower() + text[1:]
    return tpl.format(text)

def encode_ensemble(encoder, texts, templates, batch_size=64, lower_first=True):
    """Rata-rata embedding teks di bawah tiap templat, lalu normalisasi ulang. -> (n, d)"""
    outs = []
    for tpl in templates:
        wrapped = [apply_template(tpl, t, lower_first) for t in texts]
        outs.append(np.asarray(encoder.encode(wrapped, batch_size=batch_size, normalize_embeddings=True)))
    return l2norm(np.mean(outs, axis=0))

def retrieval_metrics(Q, G, q_group=None, g_group=None, ks=(1, 5, 10), map_k=10):
    nq, ng = len(Q), len(G)
    q_group = np.arange(nq) if q_group is None else np.asarray(q_group)
    g_group = np.arange(ng) if g_group is None else np.asarray(g_group)
    order = np.argsort(-(Q @ G.T), axis=1, kind="stable")
    rel = (g_group[order] == q_group[:, None])
    n_rel = (g_group[None, :] == q_group[:, None]).sum(1)
    out = {f"R@{k}": float(rel[:, :k].any(1).mean()) for k in ks}
    topk = rel[:, :map_k].astype(float)
    prec = np.cumsum(topk, 1) / np.arange(1, map_k + 1)
    out[f"mAP@{map_k}"] = float(((prec * topk).sum(1) / np.minimum(n_rel, map_k)).mean())
    return out

def four_directions(E, Ti, Te, **kw):
    return {
        "gambar->teks ID": retrieval_metrics(E, Ti, **kw),
        "gambar->teks EN": retrieval_metrics(E, Te, **kw),
        "teks ID->gambar": retrieval_metrics(Ti, E, **kw),
        "teks EN->gambar": retrieval_metrics(Te, E, **kw),
    }
print("Fungsi utilitas siap.")


Fungsi utilitas siap.


## 3 · Load cache dari Peran 2 dan CSV

Sesuaikan path. Contoh di bawah memakai path Google Drive yang sudah terbukti jalan.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/MULTIMODAL/Milestone/cache"                    # <-- sesuaikan
DATA_DIR  = "/content/drive/MyDrive/MULTIMODAL/bundle-mahasiswa/bundle-mahasiswa"  # <-- folder train.csv/val.csv

def load(split):
    z = np.load(f"{CACHE_DIR}/emb_{split}.npz")
    return {"img": z["img"], "cap_id": z["id"], "cap_en": z["en"]}

def captions(split):
    df = pd.read_csv(f"{DATA_DIR}/{split}.csv")
    return list(df["caption_id"]), list(df["caption_en"])

tr = load("train")
va = load("val")
print("train  img:", tr["img"].shape, " cap_id:", tr["cap_id"].shape, " cap_en:", tr["cap_en"].shape)
print("val    img:", va["img"].shape, " cap_id:", va["cap_id"].shape, " cap_en:", va["cap_en"].shape)


Mounted at /content/drive
train  img: (2072, 512)  cap_id: (2072, 512)  cap_en: (2072, 512)
val    img: (275, 512)  cap_id: (275, 512)  cap_en: (275, 512)


## 4 · Load encoder (butuh internet, lama di cell pertama)

In [5]:
from sentence_transformers import SentenceTransformer

img_enc = SentenceTransformer("sentence-transformers/clip-ViT-B-32")
txt_id  = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1")
print("Encoder siap.")


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

0_CLIPModel/model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

0_CLIPModel/model.safetensors: downloading bytes:           |  0.00B            

... (19 outputs left out)

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 1.57MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

Encoder siap.


## 5 · Seleksi greedy V1 (hanya train, val tidak disentuh)

Waktu seleksi disimpan ke `templates_terpilih_v2.json` supaya catatan waktu lengkap.

In [6]:
def greedy_select(encoder, texts_train, img_train, candidates, max_k=5, batch_size=64, tol=0.0):
    per = {t: np.asarray(encoder.encode([apply_template(t, x) for x in texts_train],
                                        batch_size=batch_size, normalize_embeddings=True))
           for t in candidates}
    def score(sel):
        Ti = l2norm(np.mean([per[t] for t in sel], axis=0))
        return (retrieval_metrics(Ti, img_train)["R@1"] +
                retrieval_metrics(img_train, Ti)["R@1"]) / 2
    chosen = ["{}"]
    trace = []
    best = score(chosen)
    trace.append((list(chosen), best))
    while len(chosen) < max_k:
        gains = [(score(chosen + [t]), t) for t in candidates if t not in chosen]
        if not gains: break
        s, t = max(gains)
        if s - best <= tol: break
        chosen.append(t); best = s; trace.append((list(chosen), s))
    return chosen, trace

rng = np.random.default_rng(SEED)
cap_id_train, _ = captions("train")
idx = np.sort(rng.choice(len(cap_id_train), size=min(N_SUB, len(cap_id_train)), replace=False))

t0 = time.time()
chosen, trace = greedy_select(txt_id, [cap_id_train[i] for i in idx], tr["img"][idx],
                              TEMPLATES_ID, max_k=MAX_K)
t_select = round(time.time() - t0, 1)

print(f"Selesai seleksi ({t_select} dtk)")
print("Templat terpilih V1:", chosen)
for s, r in trace:
    print(f"  {s}  ->  R@1 train rata-rata dua arah ID = {r:.4f}")

json.dump({"seed": SEED, "n_sub": int(len(idx)), "split": "train", "chosen": chosen,
           "waktu_seleksi_dtk": t_select,
           "trace": [{"set": s, "R@1_train_rata2_dua_arah_ID": r} for s, r in trace]},
          open("templates_terpilih_v2.json", "w"), ensure_ascii=False, indent=2)
print("Disimpan: templates_terpilih_v2.json")


Selesai seleksi (288.6 dtk)
Templat terpilih V1: ['{}', 'ini adalah foto {}']
  ['{}']  ->  R@1 train rata-rata dua arah ID = 0.1580
  ['{}', 'ini adalah foto {}']  ->  R@1 train rata-rata dua arah ID = 0.1720
Disimpan: templates_terpilih_v2.json


## 6 · Evaluasi V0, V1, V2, K1 di val (jalankan sekali)

In [7]:
SEL = json.load(open("templates_terpilih_v2.json"))
chosen_v1 = SEL["chosen"]
cap_id_val, cap_en_val = captions("val")

t0 = time.time()
Ti_v1 = encode_ensemble(txt_id, cap_id_val, chosen_v1)           # V1: greedy
Ti_v2 = encode_ensemble(txt_id, cap_id_val, TEMPLATES_ID)        # V2: semua 10 templat ID
Te_k1 = encode_ensemble(img_enc, cap_en_val, TEMPLATES_EN[:len(chosen_v1)])  # K1: EN, ukuran = V1
t_eval = round(time.time() - t0, 1)

results = {
    "V0 baseline M1":           four_directions(va["img"], va["cap_id"], va["cap_en"]),
    "V1 ensemble ID (greedy)":  four_directions(va["img"], Ti_v1, va["cap_en"]),
    "V2 ensemble ID (semua 10)":four_directions(va["img"], Ti_v2, va["cap_en"]),
    "K1 kontrol ensemble EN":   four_directions(va["img"], va["cap_id"], Te_k1),
}
results["_meta"] = {
    "split": "val", "seed": SEED,
    "templat_V1": chosen_v1,
    "templat_V2": TEMPLATES_ID,
    "waktu_seleksi_dtk": SEL["waktu_seleksi_dtk"],
    "waktu_eval_dtk": t_eval,
    "config": {"n_sub": N_SUB, "max_k": MAX_K,
               "encoder_img_en": "clip-ViT-B-32",
               "encoder_teks_id": "clip-ViT-B-32-multilingual-v1"},
    "parameter_dilatih": 0,
}
json.dump(results, open("hasil_m2_prompt_ensemble_v2.json", "w"), ensure_ascii=False, indent=2)
print("Disimpan: hasil_m2_prompt_ensemble_v2.json | eval", t_eval, "dtk")


Disimpan: hasil_m2_prompt_ensemble_v2.json | eval 269.8 dtk


## 7 · Tabel ringkas untuk laporan

Sanity check: baris V1 harus sama dengan run sebelumnya (G→ID R@1 = 0,2436; ID→G R@1 = 0,1891). Kalau beda, cek seed dan cache.

In [8]:
res = json.load(open("hasil_m2_prompt_ensemble_v2.json"))
base = res["V0 baseline M1"]
arah = [("teks ID->gambar", "ID->G"), ("gambar->teks ID", "G->ID"),
        ("teks EN->gambar", "EN->G"), ("gambar->teks EN", "G->EN")]

print("Varian\tarah\tR@1\tR@5\tR@10\tmAP@10\tdelta R@1 vs V0")
for v in [k for k in res if not k.startswith("_")]:
    for key, lab in arah:
        m = res[v][key]
        d = m["R@1"] - base[key]["R@1"]
        print(f"{v}\t{lab}\t{m['R@1']:.4f}\t{m['R@5']:.4f}\t{m['R@10']:.4f}\t{m['mAP@10']:.4f}\t{d:+.4f}")


Varian	arah	R@1	R@5	R@10	mAP@10	delta R@1 vs V0
V0 baseline M1	ID->G	0.1964	0.4182	0.5055	0.2830	+0.0000
V0 baseline M1	G->ID	0.2145	0.3745	0.5055	0.2924	+0.0000
V0 baseline M1	EN->G	0.3818	0.6727	0.7709	0.5073	+0.0000
V0 baseline M1	G->EN	0.4436	0.7127	0.8327	0.5576	+0.0000
V1 ensemble ID (greedy)	ID->G	0.1891	0.4036	0.5200	0.2803	-0.0073
V1 ensemble ID (greedy)	G->ID	0.2436	0.4073	0.5236	0.3201	+0.0291
V1 ensemble ID (greedy)	EN->G	0.3818	0.6727	0.7709	0.5073	+0.0000
V1 ensemble ID (greedy)	G->EN	0.4436	0.7127	0.8327	0.5576	+0.0000
V2 ensemble ID (semua 10)	ID->G	0.1964	0.4109	0.5236	0.2886	+0.0000
V2 ensemble ID (semua 10)	G->ID	0.2291	0.4291	0.5600	0.3205	+0.0145
V2 ensemble ID (semua 10)	EN->G	0.3818	0.6727	0.7709	0.5073	+0.0000
V2 ensemble ID (semua 10)	G->EN	0.4436	0.7127	0.8327	0.5576	+0.0000
K1 kontrol ensemble EN	ID->G	0.1964	0.4182	0.5055	0.2830	+0.0000
K1 kontrol ensemble EN	G->ID	0.2145	0.3745	0.5055	0.2924	+0.0000
K1 kontrol ensemble EN	EN->G	0.3818	0.6691	0.7818	0.5092	+

In [11]:
from google.colab import files
files.download("hasil_m2_prompt_ensemble_v2.json")
files.download("templates_terpilih_v2.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>